<a href="https://colab.research.google.com/github/ruforavishnu/ai_agent_elara/blob/master/eth_ai_trader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project ETH AI trader

## Step 1

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    print("Uploaded:", filename)

    os.makedirs(
        "/content/drive/MyDrive/eth-ai-trader/data/processed",
        exist_ok=True
    )

    shutil.copy(
        filename,
        "/content/drive/MyDrive/eth-ai-trader/data/processed/" + filename
    )

    print("Copied to Google Drive:")
    print(
        "/content/drive/MyDrive/eth-ai-trader/data/processed/"
        + filename
    )

Saving training_dataset.csv to training_dataset.csv
Uploaded: training_dataset.csv
Copied to Google Drive:
/content/drive/MyDrive/eth-ai-trader/data/processed/training_dataset.csv


In [3]:
import tensorflow as tf
import pandas as pd
import numpy as np

print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

TensorFlow: 2.20.0
Pandas: 2.2.2
NumPy: 2.0.2


In [4]:
df = pd.read_csv(
    "/content/training_dataset.csv"
)

print(df.shape)

df.head()

(192458, 18)


,datetime,15m_open,15m_high,15m_low,15m_close,4h_open,4h_high,4h_low,4h_close,15m_sma10,15m_sma20,15m_sma10_angle,15m_sma20_angle,15m_ao_color,4h_sma5,4h_sma10,4h_ao_color,4h_histogram_direction
0,2021-01-02 12:00:00,729.70,730.79,728.56,728.94,729.7,772.8,728.25,768.43,728.975,731.0180,-18.469319,-15.599733,red,738.51,737.433,green,rising
1,2021-01-02 12:15:00,728.94,743.00,728.25,739.94,729.7,772.8,728.25,768.43,730.428,731.2445,0.733346,-12.166735,green,738.51,737.433,green,rising
2,2021-01-02 12:30:00,739.96,750.00,736.57,750.00,729.7,772.8,728.25,768.43,732.720,732.0250,30.574594,-1.042668,green,738.51,737.433,green,rising
3,2021-01-02 12:45:00,749.99,757.00,749.81,752.38,729.7,772.8,728.25,768.43,734.741,732.8880,46.979506,12.363691,green,738.51,737.433,green,rising
4,2021-01-02 13:00:00,752.39,762.32,749.11,759.90,729.7,772.8,728.25,768.43,737.666,734.3075,59.077649,29.963449,green,738.51,737.433,green,rising


In [5]:
import numpy as np


def create_labels(df):

    labels = []

    closes = df["15m_close"].values

    ao = df["15m_ao_color"].values

    trend = df["4h_histogram_direction"].values


    for i in range(len(df)):

        label = 0   # HOLD


        if i + 20 < len(df):

            future = closes[i+1:i+20]

            current = closes[i]


            future_return = (
                future.max() - current
            ) / current


            future_drop = (
                current - future.min()
            ) / current


            # LONG opportunity

            if (
                trend[i] == "rising"
                and ao[i] == "green"
                and future_return > 0.01
            ):
                label = 1


            # SHORT opportunity

            elif (
                trend[i] == "falling"
                and ao[i] == "red"
                and future_drop > 0.01
            ):
                label = 2


        labels.append(label)


    return np.array(labels)



y = create_labels(df)


print(
    np.unique(
        y,
        return_counts=True
    )
)

(array([0, 1, 2]), array([149604,  21377,  21477]))


In [6]:
features = [

    "15m_open",
    "15m_high",
    "15m_low",
    "15m_close",

    "4h_open",
    "4h_high",
    "4h_low",
    "4h_close",

    "15m_sma10",
    "15m_sma20",

    "15m_sma10_angle",
    "15m_sma20_angle",

    "4h_sma5",
    "4h_sma10",
]


X = df[features].values


print(X.shape)

(192458, 14)


In [7]:
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()

X = scaler.fit_transform(X)


print(X.shape)

(192458, 14)


In [8]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)


print(
    X_train.shape,
    X_test.shape
)

(153966, 14) (38492, 14)


In [9]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout


model = Sequential([

    Dense(
        64,
        activation="relu",
        input_shape=(X_train.shape[1],)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        3,
        activation="softmax"
    )

])


model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]

)


model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,139 (12.26 KB)

 Trainable params: 3,139 (12.26 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_test,
        y_test
    ),

    epochs=30,

    batch_size=256

)

Epoch 1/30
602/602 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.7714 - loss: 0.6521 - val_accuracy: 0.8029 - val_loss: 0.5629
Epoch 2/30
602/602 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7779 - loss: 0.5909 - val_accuracy: 0.8126 - val_loss: 0.4911
Epoch 3/30
602/602 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7919 - loss: 0.5241 - val_accuracy: 0.8278 - val_loss: 0.4346
Epoch 4/30
602/602 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8029 - loss: 0.4839 - val_accuracy: 0.8333 - val_loss: 0.4083
Epoch 5/30
602/602 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8089 - loss: 0.4681 - val_accuracy: 0.8356 - val_loss: 0.4070
Epoch 6/30
602/602 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8119 - loss: 0.4580 - val_accuracy: 0.8421 - val_loss: 0.3916
Epoch 7/30
602/602 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8136 - loss: 0.4540 - val_accuracy: 0.8428 - val_loss: 0.3922
Epoch 8/30
602/602 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8151 - loss: 0.4491 - val_accuracy: 0.

## Now save the deep learning ai model, so we dont have to re-train it agai when the google colab session expires

In [11]:
model_path = "/content/drive/MyDrive/eth-ai-trader/models/eth_ai_model.keras"

model.save(model_path)

print("MODEL SAVED")
print(model_path)

MODEL SAVED
/content/drive/MyDrive/eth-ai-trader/models/eth_ai_model.keras


## No re-import the saved model. so next time when running you have to run only from this point. sice the dl model is already saved.

In [12]:
import joblib

scaler_path = "/content/drive/MyDrive/eth-ai-trader/models/scaler.pkl"

joblib.dump(
    scaler,
    scaler_path
)

print("SCALER SAVED")
print(scaler_path)

SCALER SAVED
/content/drive/MyDrive/eth-ai-trader/models/scaler.pkl


In [13]:
import os

print(
    os.listdir(
        "/content/drive/MyDrive/eth-ai-trader/models"
    )
)

['__init__.py', 'eth_ai_model.keras', 'scaler.pkl']
